# TD 7 : classer des avis de la HAS, et savoir d'où vient le score

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. construire un classifieur de texte en quelques lignes
2. reconnaître un score trop beau et en trouver la cause dans les données
3. mesurer ce qu'un score devait à une seule feature
4. lire un rapport par classe et dire ce que l'accuracy cachait

## Prérequis

- avoir fini les TD 1 à 6
- avoir suivi le bloc A de ce matin

**Durée :** 105 minutes

## Le contexte

La Haute Autorité de Santé publie un avis pour chaque médicament remboursé, avec un niveau de service médical rendu. 15 315 avis en français, en open data. Peut-on prédire le niveau à partir du texte ?

## Le déroulé

| quand | quoi |
|---|---|
| 4:00 | étape 1 : le corpus et le baseline |
| 4:20 | étape 2 : le classifieur, et la joie |
| 4:40 | étape 3 : d'où vient ce score ? |
| 5:10 | étape 4 : la vraie tâche |
| 5:45 | pause |

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

---

## Étape 1 : Le corpus et le baseline

**20 minutes.** Le réflexe de la journée 1 s'applique au texte sans changer d'un mot.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# charger et filtrer sur les quatre niveaux principaux
avis = pd.read_parquet(DONNEES / 'avis_has.parquet')
avis = avis[avis['smr'].isin(NIVEAUX)]

# vectoriser du texte
from sklearn.feature_extraction.text import TfidfVectorizer
vect = TfidfVectorizer(min_df=3, ngram_range=(1, 2))

# un pipeline vectoriseur plus modele
from sklearn.pipeline import make_pipeline
modele = make_pipeline(vect, LogisticRegression(max_iter=2000))

# accuracy et macro F1
from sklearn.metrics import accuracy_score, f1_score
f1_score(y_te, predit, average='macro')

# le rapport par classe
from sklearn.metrics import classification_report
print(classification_report(y_te, predit, digits=3))

# remplacer un motif dans du texte
import re
re.sub(r'motif', 'REMPLACEMENT', texte, flags=re.I)
```

### 1.1 Charger, filtrer, et regarder la répartition

In [ ]:
# A FAIRE : charger les avis, garder les quatre niveaux principaux,
#           retirer les textes de moins de 40 caracteres.
NIVEAUX = ['Important', 'Insuffisant', 'Modéré', 'Faible']

avis = ...

print(f'{len(avis):,} avis retenus')
print(avis['smr'].value_counts())
print(f"\nlongueur mediane du texte : {int(avis['texte'].str.len().median())} car.")

**Indice 1.** Trois étapes enchaînées : `read_parquet`, un filtre `isin(NIVEAUX)` sur `smr`, et un filtre `str.len() >= 40` sur `texte`. Pensez à `.strip()`.

**Indice 2.** `avis = pd.read_parquet(DONNEES / 'avis_has.parquet')` puis `avis = avis[avis['smr'].isin(NIVEAUX)].copy()`, `avis['texte'] = avis['texte'].str.strip()`, `avis = avis[avis['texte'].str.len() >= 40]`

In [ ]:
assert len(avis) == 15315, f'attendu 15 315 avis, obtenu {len(avis):,}'
part = avis['smr'].value_counts(normalize=True)
assert abs(part['Important'] - 0.629) < 0.01
print(f'correct. La classe majoritaire pese {part["Important"]:.1%}.')
print(f'Donc répondre toujours Important donne {part["Important"]:.2%}')
print('d accuracy sans rien apprendre. C est le chiffre a battre.')

### 1.2 Le baseline, et pourquoi deux métriques

In [ ]:
# A FAIRE : le split, puis le baseline, avec accuracy ET macro F1.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    avis['texte'].values, avis['smr'].values,
    test_size=0.25, random_state=42, stratify=avis['smr'].values)

bidon = ...
predit_bidon = ...

print(f'baseline  accuracy {accuracy_score(y_te, predit_bidon):.4f}'
      f'   macro F1 {f1_score(y_te, predit_bidon, average="macro"):.4f}')

**Indice 1.** `DummyClassifier(strategy='most_frequent')` s'entraîne et prédit comme tout modèle sklearn.

**Indice 2.** `bidon = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)` puis `predit_bidon = bidon.predict(X_te)`

In [ ]:
acc_b = accuracy_score(y_te, predit_bidon)
f1_b = f1_score(y_te, predit_bidon, average='macro')
assert abs(acc_b - 0.6289) < 0.01
print(f'accuracy {acc_b:.4f}, macro F1 {f1_b:.4f}.')
print()
print('Regardez l ecart entre les deux. L accuracy est flatteuse parce qu une')
print('classe pese 63 %. Le macro F1 pese les quatre classes a egalite,')
print('donc il s effondre quand trois d entre elles ne sont jamais predites.')
print('Sur des classes desequilibrees, donnez toujours les deux.')

---

## Étape 2 : Le classifieur

**20 minutes.** Trois lignes, et un résultat spectaculaire.

In [ ]:
# A FAIRE : TF-IDF plus logistic regression, dans un pipeline.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

modele = ...
predit = ...

acc = accuracy_score(y_te, predit)
f1 = f1_score(y_te, predit, average='macro')
print(f'TF-IDF  accuracy {acc:.4f}   macro F1 {f1:.4f}')
print(f'baseline                 {acc_b:.4f}             {f1_b:.4f}')

**Indice 1.** Le patron du mémo : `make_pipeline(vect, LogisticRegression(max_iter=2000))`, puis `.fit(X_tr, y_tr)` et `.predict(X_te)`.

**Indice 2.** `modele = make_pipeline(TfidfVectorizer(min_df=3, ngram_range=(1, 2)), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)`

In [ ]:
assert acc > 0.98, f'attendu plus de 0,98, obtenu {acc:.4f}'
print(f'{acc:.2%} d accuracy sur du texte clinique réel, en francais,')
print(f'contre {acc_b:.2%} pour le baseline. Et en moins d une seconde.')
print()
print('Prenez trente secondes pour trouver cela suspect.')
print('Puis passez a l etape 3.')

---

## Étape 3 : D'où vient ce score ?

**30 minutes.** La question à poser devant tout résultat trop beau. Elle se répond en lisant.

### 3.1 Lisez trois avis

Aucune métrique ne vous dira ce qui suit. Il faut regarder les données.

In [ ]:
# A FAIRE : afficher trois avis du jeu de test avec leur niveau.
#           Lisez-les vraiment, mot par mot.
for i in range(3):
    ...

**Indice 1.** Parcourez `zip(X_te, y_te)` et affichez le niveau puis les 200 premiers caractères du texte.

**Indice 2.** `for texte, niveau in list(zip(X_te, y_te))[:3]: print(niveau, '|', texte[:200])`

In [ ]:
import re
MOTS = re.compile(r'important|mod[ée]r[ée]|insuffisant|faible', re.I)
part = np.mean([bool(MOTS.search(t)) for t in avis['texte']])
print(f'Part des avis contenant un mot de niveau : {part:.1%}')
print()
print('Le mot du label est dans le texte. Presque toujours.')
print('La phrase type est : le service medical rendu par X est IMPORTANT')
print('dans l indication de l AMM.')
assert part > 0.95, 'le mot devrait etre present presque partout'

### 3.2 Prouvez-le : une règle de quatre lignes

Si le modèle ne fait que repérer un mot, alors un code qui repère ce mot devrait faire aussi bien. **Écrivez-le.**

In [ ]:
# A FAIRE : un classifieur sans aucun apprentissage.
#           Il cherche les quatre mots et rend le niveau correspondant.
def regle(texte):
    ...

predit_regle = np.array([regle(t) for t in X_te])
print(f'regle   accuracy {accuracy_score(y_te, predit_regle):.4f}'
      f'   macro F1 {f1_score(y_te, predit_regle, average="macro"):.4f}')
print(f'modele  accuracy {acc:.4f}   macro F1 {f1:.4f}')

**Indice 1.** Passez le texte en minuscules, cherchez chaque mot de niveau à son tour, et rendez le niveau correspondant. Attention à `modéré` qui peut être écrit sans accent.

**Indice 2.** Testez dans l'ordre : `insuffisant`, puis `modér`, puis `faible`, puis `important`. Rendez `'Important'` par défaut.

In [ ]:
acc_regle = accuracy_score(y_te, predit_regle)
assert acc_regle > 0.95, (
    f'la regle devrait faire presque aussi bien, obtenu {acc_regle:.4f}')
print(f'ecart entre la regle et le modele entraine : '
      f'{acc - acc_regle:+.4f}')
print()
print('Quatre lignes de code, aucun apprentissage, aucune donnee.')
print('Le modele n a rien appris qu une expression reguliere ne trouvait.')
print()
print('Et la regle est plus rapide, plus simple, et explicable.')
print('Si la tache etait reellement celle-la, il faudrait livrer la regle.')

---

## Étape 4 : La vraie tâche

**35 minutes.** Masquer le mot, remesurer, et découvrir ce que la tâche valait vraiment.

### 4.1 Masquer, et remesurer

On ne supprime pas le mot : on le **remplace** par un jeton neutre. Supprimer raccourcirait les phrases, ce qui serait une seconde différence et rendrait la comparaison moins propre.

In [ ]:
# A FAIRE : masquer les mots de niveau, puis reentrainer a l identique.
def masquer(texte):
    ...

Xm_tr = np.array([masquer(t) for t in X_tr])
Xm_te = np.array([masquer(t) for t in X_te])

modele_m = ...
predit_m = ...

acc_m = accuracy_score(y_te, predit_m)
f1_m = f1_score(y_te, predit_m, average='macro')
print(f'texte masque  accuracy {acc_m:.4f}   macro F1 {f1_m:.4f}')
print(f'texte brut    accuracy {acc:.4f}   macro F1 {f1:.4f}')
print(f'baseline      accuracy {acc_b:.4f}   macro F1 {f1_b:.4f}')

**Indice 1.** `MOTS.sub('NIVEAU', texte)` remplace tous les mots de niveau d'un coup. Le modèle est exactement le même qu'à l'étape 2.

**Indice 2.** `def masquer(texte): return MOTS.sub('NIVEAU', texte)`, puis le même `make_pipeline(...)` entraîné sur `Xm_tr`.

In [ ]:
assert abs(acc_m - 0.8809) < 0.02, (
    f'attendu environ 0,8809, obtenu {acc_m:.4f}')
assert acc_m > acc_b, (
    'apres masquage le modele doit rester au-dessus du baseline : '
    'il y a du signal réel dans le texte')
print(f'l accuracy perd {acc - acc_m:.4f}')
print(f'le macro F1 perd {f1 - f1_m:.4f}, soit presque le double')
print()
print('Deux lectures, et il faut les deux :')
print(f'  1. l ecart mesure ce que le premier score devait a UN MOT')
print(f'  2. et {acc_m:.2%} reste tres au-dessus de {acc_b:.2%} : il y a')
print('     donc du signal réel dans la formulation des avis.')
print()
print('La tache existe. Elle est seulement beaucoup moins spectaculaire.')

### 4.2 Et maintenant, regardez par classe

In [ ]:
# A FAIRE : le rapport par classe du modele sur texte masque.
from sklearn.metrics import classification_report, confusion_matrix

print(...)

cm = confusion_matrix(y_te, predit_m, labels=NIVEAUX)
print(f"{'':<14}" + ''.join(f'{n[:11]:>13}' for n in NIVEAUX))
for i, n in enumerate(NIVEAUX):
    print(f'{n:<14}' + ''.join(f'{cm[i][j]:>13,}' for j in range(4)))

**Indice 1.** `classification_report(y_te, predit_m, digits=3)` donne précision, rappel et F1 par classe.

**Indice 2.** `print(classification_report(y_te, predit_m, digits=3, zero_division=0))`

In [ ]:
from sklearn.metrics import recall_score
rappels = recall_score(y_te, predit_m, labels=NIVEAUX, average=None,
                       zero_division=0)
for niveau, r in zip(NIVEAUX, rappels):
    print(f'   rappel {niveau:<14} {r:.3f}')
assert rappels[0] > 0.95, 'Important devrait avoir un tres bon rappel'
assert rappels[3] < 0.70, 'Faible devrait avoir un rappel mediocre'
print()
print(f'Le modele trouve {rappels[0]:.1%} des avis Important')
print(f'et seulement {rappels[3]:.1%} des avis Faible.')
print()
print('Regardez la colonne Important de la matrice : les erreurs des trois')
print('autres classes y tombent presque toutes. Tout glisse vers la classe')
print('majoritaire, parce que c est ce qui maximise l accuracy.')
print()
print('Et c est la classe rare qui compte : un SMR faible ou insuffisant')
print('declenche une decision de deremboursement.')

## Ce que cette étape a établi

- Le réflexe du taux de base s'applique au texte : 62,9 % ici.
- TF-IDF plus logistic regression donne 0,9932, et une règle de quatre lignes 0,9906. Le modèle n'avait rien appris de plus qu'une regex.
- 99,8 % des avis contiennent le mot de leur propre niveau. Aucune métrique ne l'aurait dit : il a fallu lire.
- Une fois le mot masqué : 0,8809 d'accuracy et 0,8030 de macro F1. La tâche existe, elle est simplement plus modeste.
- Et le détail par classe montre un rappel de 0,992 sur Important contre 0,532 sur Faible, la classe qui décide d'un déremboursement.

## Les erreurs classiques

- **Se réjouir d'un score inattendu.** Un score surprenant est une question, pas une victoire.
- **Modéliser sans avoir lu dix exemples.** Dix minutes de lecture vous auraient évité toute la séance.
- **Donner l'accuracy seule sur des classes déséquilibrées.** Donnez le macro F1 à côté, et le rapport par classe si la décision en dépend.

## Pour aller plus loin, si vous avez fini en avance

- Entraîner sur le texte masqué en ne gardant que les avis « Faible » et « Modéré ». Ces deux-là sont-ils distinguables entre eux, ou le modèle ne savait-il que les séparer d'« Important » ?
- Regarder les coefficients du modèle masqué : quels mots porte-t-il maintenant ? `modele_m.named_steps` donne accès au vectoriseur et au modèle.